# Chapter 3: Probability & Information Theory for LLMs

**Mathematics Behind LLMs — Book Series**

This chapter covers the probabilistic and information-theoretic foundations that underpin every large language model: how LLMs define distributions over sequences, how we measure information and uncertainty, and how these ideas translate directly into training objectives and decoding strategies.

---

**Topics covered:**
1. LLMs as Probability Distributions
2. Conditional Probability & Bayes' Theorem
3. Maximum Likelihood Estimation (MLE)
4. Shannon Entropy
5. Cross-Entropy Loss & Perplexity
6. KL Divergence
7. Softmax & Temperature Scaling
8. Top-k & Top-p (Nucleus) Sampling
9. Perplexity Evaluation

In [ ]:
# Setup: imports and reproducibility
import torch
import torch.nn as nn
import torch.nn.functional as F
import math

torch.manual_seed(42)

print(f"PyTorch version: {torch.__version__}")
print("Seed set to 42 for reproducibility.")

## 3.1 LLMs as Probability Distributions

A language model defines a probability distribution over sequences of tokens. Using the **chain rule of probability**, any joint distribution over a sequence $x_1, x_2, \ldots, x_T$ can be factored as:

$$P(x_1, x_2, \ldots, x_T) = \prod_{t=1}^{T} P(x_t \mid x_{<t})$$

where $x_{<t} = (x_1, \ldots, x_{t-1})$ is the context prefix.

**At each step**, the transformer's forward pass outputs a vector of **logits** $\mathbf{z} \in \mathbb{R}^{|V|}$ over the vocabulary $V$. Applying softmax converts logits to probabilities:

$$P(x_t = v \mid x_{<t}) = \frac{e^{z_v}}{\sum_{v'} e^{z_{v'}}}$$

The model then **samples** or **greedily selects** the next token from this distribution.

**Key insight:** Training minimizes the negative log-likelihood $-\log P(x_t \mid x_{<t})$ summed over all positions — i.e., the cross-entropy between the true next token (one-hot) and the predicted distribution.

In [ ]:
# Simulate next-token probability distribution over a small vocabulary

torch.manual_seed(42)
vocab_size = 10  # small vocabulary for illustration

# Simulate logits from a toy LM forward pass
logits = torch.randn(vocab_size)  # shape: (V,)
print("Logits (raw scores):", logits.round(decimals=3))

# Convert to probabilities via softmax
probs = F.softmax(logits, dim=-1)
print("\nProbabilities (softmax):", probs.round(decimals=4))
print(f"Sum of probabilities: {probs.sum().item():.6f}  (should be 1.0)")

# Sample next token using torch.distributions.Categorical
dist = torch.distributions.Categorical(probs=probs)
sampled_tokens = [dist.sample().item() for _ in range(10)]
print(f"\n10 sampled token ids: {sampled_tokens}")

# Greedy decoding: pick argmax
greedy_token = probs.argmax().item()
print(f"Greedy token id: {greedy_token}  (prob = {probs[greedy_token]:.4f})")

# Log-probabilities (more numerically stable)
log_probs = F.log_softmax(logits, dim=-1)
print(f"\nLog-prob of greedy token: {log_probs[greedy_token]:.4f}")
print(f"Sequence log-prob example (3 tokens): {log_probs[greedy_token].item():.4f} + ...")

## 3.2 Conditional Probability & Bayes' Theorem

**Conditional probability** formalizes how the probability of an event changes given knowledge of another event:

$$P(A \mid B) = \frac{P(A \cap B)}{P(B)}$$

**Bayes' Theorem** inverts the conditioning, allowing inference from observations to parameters:

$$P(\theta \mid X) = \frac{P(X \mid \theta)\, P(\theta)}{P(X)}$$

- $P(\theta \mid X)$: **posterior** — updated belief about parameters after seeing data
- $P(X \mid \theta)$: **likelihood** — how well the model explains the data
- $P(\theta)$: **prior** — belief about parameters before seeing data
- $P(X)$: **evidence** (normalizing constant)

In LLM pretraining, we maximize the likelihood $P(X \mid \theta)$ (MLE), implicitly using a flat prior. RLHF introduces an explicit KL penalty that acts as a prior toward the reference model.

**Joint probability from a count matrix:** Given a co-occurrence table $N$, we estimate:

$$P(x_i, x_j) = \frac{N_{ij}}{\sum_{i,j} N_{ij}}, \quad P(x_j \mid x_i) = \frac{N_{ij}}{\sum_j N_{ij}}$$

In [ ]:
# Conditional probability from a bigram count matrix

# 3x3 bigram counts: rows = current token, cols = next token
# Tokens: 0='the', 1='cat', 2='sat'
count_matrix = torch.tensor([
    [5, 10, 3],   # 'the' followed by: 'the'=5, 'cat'=10, 'sat'=3
    [2,  1, 8],   # 'cat' followed by: 'the'=2, 'cat'=1, 'sat'=8
    [7,  0, 2],   # 'sat' followed by: 'the'=7, 'cat'=0, 'sat'=2
], dtype=torch.float32)

vocab = ['the', 'cat', 'sat']

# Joint probability P(x_i, x_j)
total = count_matrix.sum()
joint_prob = count_matrix / total
print("Joint probability P(x_i, x_j):")
for i, row in enumerate(joint_prob):
    for j, val in enumerate(row):
        print(f"  P({vocab[i]}, {vocab[j]}) = {val:.4f}")

# Marginal probability P(x_i)
marginal = joint_prob.sum(dim=1)
print("\nMarginal probability P(x_i):")
for i, p in enumerate(marginal):
    print(f"  P({vocab[i]}) = {p:.4f}")

# Conditional probability P(x_j | x_i)
row_sums = count_matrix.sum(dim=1, keepdim=True)
cond_prob = count_matrix / row_sums
print("\nConditional probability P(x_j | x_i):")
for i, row in enumerate(cond_prob):
    print(f"  P(next | '{vocab[i]}'): {dict(zip(vocab, row.tolist()))}")

# Verify Bayes: P(the | cat) = P(cat, the) / P(cat)
p_cat_the = joint_prob[1, 0].item()
p_cat = marginal[1].item()
p_the_given_cat_bayes = p_cat_the / p_cat
p_the_given_cat_direct = cond_prob[1, 0].item()
print(f"\nBayes check P('the' | 'cat'):")
print(f"  Via Bayes:  P(cat,the)/P(cat) = {p_cat_the:.4f}/{p_cat:.4f} = {p_the_given_cat_bayes:.4f}")
print(f"  Direct:     {p_the_given_cat_direct:.4f}")
print(f"  Match: {abs(p_the_given_cat_bayes - p_the_given_cat_direct) < 1e-6}")

## 3.3 Maximum Likelihood Estimation (MLE)

**MLE** finds parameters $\theta$ that maximize the probability of the observed data:

$$\hat{\theta} = \arg\max_{\theta} \prod_{i=1}^{n} p(x_i; \theta) = \arg\max_{\theta} \sum_{i=1}^{n} \log p(x_i; \theta)$$

Taking the log converts the product to a sum (numerically stable) and doesn't change the argmax since log is monotone.

**For a Gaussian** $\mathcal{N}(\mu, \sigma^2)$, the log-likelihood is:

$$\ell(\mu, \sigma^2) = -\frac{n}{2}\log(2\pi\sigma^2) - \frac{1}{2\sigma^2}\sum_{i=1}^n (x_i - \mu)^2$$

Setting $\partial\ell/\partial\mu = 0$ and $\partial\ell/\partial\sigma^2 = 0$ gives the **MLE estimates**:

$$\hat{\mu} = \bar{x} = \frac{1}{n}\sum_{i=1}^n x_i$$

$$\hat{\sigma}^2 = \frac{1}{n}\sum_{i=1}^n (x_i - \bar{x})^2$$

Note: the MLE variance uses $1/n$ (biased), not $1/(n-1)$ (unbiased/Bessel's correction).

**LLM connection:** Pretraining LLMs is exactly MLE over the training corpus. Each gradient step maximizes $\sum_t \log P(x_t \mid x_{<t}; \theta)$.

In [ ]:
# Maximum Likelihood Estimation for a Gaussian

torch.manual_seed(42)

# True parameters
true_mu = 3.5
true_sigma = 1.2
n = 1000

# Generate samples from N(mu, sigma^2)
samples = torch.normal(mean=true_mu, std=true_sigma, size=(n,))
print(f"Generated {n} samples from N({true_mu}, {true_sigma}^2)")

# MLE estimates
mu_hat = torch.mean(samples)
# MLE variance: 1/n * sum((x - mu)^2)  [biased]
sigma2_hat_mle = torch.mean((samples - mu_hat) ** 2)
sigma_hat_mle = torch.sqrt(sigma2_hat_mle)

# Unbiased estimate (Bessel's correction): 1/(n-1)
sigma2_hat_unbiased = torch.var(samples, unbiased=True)  # ddof=1
sigma_hat_unbiased = torch.sqrt(sigma2_hat_unbiased)

print(f"\nTrue mu:    {true_mu:.4f}")
print(f"MLE mu_hat: {mu_hat.item():.4f}")
print(f"\nTrue sigma:         {true_sigma:.4f}")
print(f"MLE sigma_hat:      {sigma_hat_mle.item():.4f}  (biased,  1/n)")
print(f"Unbiased sigma_hat: {sigma_hat_unbiased.item():.4f}  (unbiased, 1/(n-1))")

# Compute log-likelihood at MLE estimates
def gaussian_log_likelihood(x, mu, sigma):
    """Log-likelihood of data x under N(mu, sigma^2)."""
    n = x.shape[0]
    ll = -0.5 * n * math.log(2 * math.pi) \
         - n * torch.log(sigma) \
         - 0.5 / (sigma ** 2) * ((x - mu) ** 2).sum()
    return ll

ll_mle = gaussian_log_likelihood(samples, mu_hat, sigma_hat_mle)
ll_true = gaussian_log_likelihood(samples, torch.tensor(true_mu), torch.tensor(true_sigma))
print(f"\nLog-likelihood at MLE estimates: {ll_mle.item():.2f}")
print(f"Log-likelihood at true params:   {ll_true.item():.2f}")
print(f"MLE >= true (as expected): {ll_mle.item() >= ll_true.item()}")

## 3.4 Shannon Entropy

**Entropy** measures the average uncertainty (or information content) of a random variable:

$$H(X) = -\sum_{i} p_i \log_2 p_i$$

(using $0 \log 0 = 0$ by convention)

**Key properties:**
- **Uniform distribution** → **maximum entropy**: $H = \log_2 |V|$ bits
- **One-hot / deterministic** → **zero entropy**: $H = 0$
- Entropy is always $\geq 0$
- For $|V|$ outcomes: $0 \leq H \leq \log_2 |V|$

**Connection to LLMs:**

- High-entropy output distribution = uncertain/creative model (spread probability)
- Low-entropy output distribution = confident model (peaked probability)
- **Temperature $\tau$** controls entropy: $\text{softmax}(\mathbf{z}/\tau)$
  - $\tau \to 0$: distribution collapses to argmax (zero entropy, greedy)
  - $\tau \to \infty$: distribution approaches uniform (max entropy)
  - $\tau = 1$: standard softmax

The entropy of the model's predicted distribution at each step is a measure of the model's confidence.

In [ ]:
# Shannon Entropy: uniform, peaked, and one-hot distributions

def entropy(probs, base=2, eps=1e-10):
    """Compute Shannon entropy H(X) = -sum(p * log_b(p))."""
    probs = probs.clamp(min=eps)  # avoid log(0)
    if base == 2:
        log_fn = torch.log2
    else:
        log_fn = torch.log
    return -(probs * log_fn(probs)).sum().item()

vocab_size = 8

# 1. Uniform distribution
uniform = torch.ones(vocab_size) / vocab_size
H_uniform = entropy(uniform)
H_max = math.log2(vocab_size)
print(f"Uniform distribution (V={vocab_size}):")
print(f"  H = {H_uniform:.4f} bits  (max = log2({vocab_size}) = {H_max:.4f} bits)")

# 2. Peaked distribution (one token much more likely)
logits_peaked = torch.tensor([5.0, 1.0, 0.5, 0.2, 0.1, 0.1, 0.05, 0.05])
peaked = F.softmax(logits_peaked, dim=0)
H_peaked = entropy(peaked)
print(f"\nPeaked distribution (max prob = {peaked.max():.3f}):")
print(f"  H = {H_peaked:.4f} bits")

# 3. One-hot (deterministic)
one_hot = torch.zeros(vocab_size)
one_hot[0] = 1.0
H_one_hot = entropy(one_hot)
print(f"\nOne-hot distribution:")
print(f"  H = {H_one_hot:.4f} bits  (minimum: 0)")

# Temperature effect on entropy
print("\n--- Temperature effect on entropy ---")
base_logits = torch.tensor([3.0, 1.5, 1.0, 0.5, 0.3, 0.2, 0.1, 0.0])
for tau in [0.1, 0.5, 1.0, 2.0, 10.0]:
    probs_tau = F.softmax(base_logits / tau, dim=0)
    H_tau = entropy(probs_tau)
    print(f"  tau={tau:4.1f}:  H = {H_tau:.4f} bits  "
          f"(max_prob = {probs_tau.max():.4f})")

## 3.5 Cross-Entropy Loss

**Cross-entropy** measures the expected number of bits needed to encode samples from distribution $P$ using a code optimized for distribution $Q$:

$$H(P, Q) = -\sum_i P(i) \log Q(i)$$

For language modeling with one-hot targets (true next token $y$):

$$\mathcal{L}_{CE} = -\sum_i y_i \log \hat{p}_i = -\log \hat{p}_y$$

Over a full sequence of $T$ tokens:

$$\mathcal{L} = -\frac{1}{T} \sum_{t=1}^{T} \log P(x_t \mid x_{<t})$$

**Perplexity (PPL)** is the exponential of the average cross-entropy:

$$\text{PPL} = \exp(\mathcal{L}) = \exp\!\left(-\frac{1}{T}\sum_t \log P(x_t \mid x_{<t})\right)$$

PPL can be interpreted as the **effective vocabulary size** at each step — a model with PPL=50 is as uncertain as a uniform distribution over 50 options.

**Label smoothing** replaces the one-hot target with a soft distribution:

$$\tilde{y}_i = \begin{cases} 1 - \epsilon + \epsilon/|V| & i = y \\ \epsilon/|V| & i \neq y \end{cases}$$

This prevents overconfidence and improves calibration.

In [ ]:
# Cross-entropy loss, perplexity, and label smoothing

torch.manual_seed(42)

batch_size = 4
seq_len = 6
vocab_size = 20

# Toy logits: (B, T, V)
logits = torch.randn(batch_size, seq_len, vocab_size)
# Target token ids: (B, T)
targets = torch.randint(0, vocab_size, (batch_size, seq_len))

# CrossEntropyLoss (standard, no smoothing)
# nn.CrossEntropyLoss expects (N, C) or (N, C, d1, ...) logits and (N,) targets
criterion = nn.CrossEntropyLoss()
# Reshape: (B*T, V) and (B*T,)
loss_standard = criterion(logits.view(-1, vocab_size), targets.view(-1))
print(f"Cross-entropy loss (standard): {loss_standard.item():.4f}")

# Perplexity
ppl = torch.exp(loss_standard)
print(f"Perplexity: {ppl.item():.2f}  (vocab={vocab_size}, random baseline={vocab_size})")

# Label smoothing
criterion_smooth = nn.CrossEntropyLoss(label_smoothing=0.1)
loss_smooth = criterion_smooth(logits.view(-1, vocab_size), targets.view(-1))
print(f"\nCross-entropy loss (label_smoothing=0.1): {loss_smooth.item():.4f}")
print(f"Perplexity with smoothing: {torch.exp(loss_smooth).item():.2f}")

# Manual cross-entropy verification
log_probs = F.log_softmax(logits, dim=-1)  # (B, T, V)
# Gather log-prob of true token at each position
target_log_probs = log_probs.gather(dim=-1, index=targets.unsqueeze(-1)).squeeze(-1)  # (B, T)
manual_loss = -target_log_probs.mean()
print(f"\nManual cross-entropy: {manual_loss.item():.4f}")
print(f"Match with nn.CrossEntropyLoss: {abs(manual_loss.item() - loss_standard.item()) < 1e-5}")

# Show loss for a 'perfect' prediction
print("\n--- Loss for varying prediction confidence ---")
for confidence in [0.1, 0.3, 0.5, 0.8, 0.95, 0.99]:
    remainder = (1.0 - confidence) / (vocab_size - 1)
    p = torch.full((vocab_size,), remainder)
    p[0] = confidence  # correct token gets 'confidence'
    ce = -math.log(confidence)
    print(f"  P(correct)={confidence:.2f}: CE={ce:.4f}, PPL={math.exp(ce):.2f}")

## 3.6 KL Divergence

**KL divergence** (Kullback-Leibler divergence) measures how distribution $P$ differs from reference distribution $Q$:

$$D_{KL}(P \| Q) = \sum_i P(i) \log \frac{P(i)}{Q(i)} \geq 0$$

**Key properties:**
- **Non-negative:** $D_{KL}(P \| Q) \geq 0$, with equality iff $P = Q$ (Gibbs' inequality)
- **Asymmetric:** $D_{KL}(P \| Q) \neq D_{KL}(Q \| P)$ in general
- Not a true distance metric (no triangle inequality)

**Relationship to cross-entropy and entropy:**

$$D_{KL}(P \| Q) = H(P, Q) - H(P)$$

**RLHF application:** In reinforcement learning from human feedback, the policy $\pi_\theta$ is trained to maximize reward while staying close to the reference (SFT) policy $\pi_{\text{ref}}$:

$$\mathcal{L}_{\text{RLHF}} = \mathbb{E}_{y \sim \pi_\theta}[r(x, y)] - \beta \cdot D_{KL}(\pi_\theta \| \pi_{\text{ref}})$$

The KL penalty prevents reward hacking — the model can't deviate too far from the base model. $\beta$ controls the tradeoff.

In [ ]:
# KL Divergence: asymmetry demo and RLHF penalty

torch.manual_seed(42)
vocab_size = 8

# Define two distributions P and Q
logits_p = torch.tensor([3.0, 1.0, 0.5, 0.2, 0.1, 0.05, 0.02, 0.01])
logits_q = torch.tensor([1.0, 2.0, 1.5, 0.8, 0.5, 0.3, 0.2, 0.1])

P = F.softmax(logits_p, dim=0)
Q = F.softmax(logits_q, dim=0)

print("P distribution:", P.round(decimals=4).tolist())
print("Q distribution:", Q.round(decimals=4).tolist())

# F.kl_div expects log-probabilities for input, probabilities for target
# F.kl_div(log_Q, P) computes sum(P * (log(P) - log(Q))) = KL(P||Q)
kl_PQ = F.kl_div(Q.log(), P, reduction='sum')  # KL(P || Q)
kl_QP = F.kl_div(P.log(), Q, reduction='sum')  # KL(Q || P)

print(f"\nKL(P || Q) = {kl_PQ.item():.6f}")
print(f"KL(Q || P) = {kl_QP.item():.6f}")
print(f"Asymmetric: KL(P||Q) != KL(Q||P)? {abs(kl_PQ.item() - kl_QP.item()) > 1e-4}")
print(f"Both >= 0? P||Q: {kl_PQ.item():.6f} >= 0: {kl_PQ.item() >= 0}")
print(f"           Q||P: {kl_QP.item():.6f} >= 0: {kl_QP.item() >= 0}")

# Verify manually
kl_manual = (P * (P.log() - Q.log())).sum()
print(f"\nManual KL(P||Q):    {kl_manual.item():.6f}")
print(f"F.kl_div KL(P||Q):  {kl_PQ.item():.6f}")

# KL = H(P,Q) - H(P)
H_P = -(P * P.log()).sum()  # entropy of P (nats)
H_PQ = -(P * Q.log()).sum()  # cross-entropy H(P,Q)
kl_from_entropy = H_PQ - H_P
print(f"\nKL from H(P,Q)-H(P): {kl_from_entropy.item():.6f}")

# RLHF KL penalty demo
print("\n--- RLHF KL Penalty Demo ---")
beta = 0.1  # KL coefficient
# Simulate rewards for 5 responses (higher = better)
rewards = torch.tensor([2.5, 1.8, 3.0, 0.5, 1.2])
# Simulate per-token KL divergences for each response
kl_penalties = torch.tensor([0.2, 0.5, 1.5, 0.05, 0.3])  # higher = more drift

rlhf_objective = rewards - beta * kl_penalties
print(f"Rewards:         {rewards.tolist()}")
print(f"KL penalties:    {kl_penalties.tolist()}")
print(f"RLHF objective (r - beta*KL): {rlhf_objective.round(decimals=3).tolist()}")
print(f"Best response by RLHF: {rlhf_objective.argmax().item()} (reward={rewards[rlhf_objective.argmax()].item()})")

## 3.7 Softmax & Temperature Scaling

**Softmax** converts raw logits to a valid probability distribution:

$$\text{softmax}(\mathbf{z})_i = \frac{e^{z_i}}{\sum_j e^{z_j}}$$

**Numerical stability — Log-Sum-Exp trick:**

For large $z_i$, $e^{z_i}$ overflows float32. The solution: subtract the maximum before exponentiating:

$$\text{softmax}(\mathbf{z})_i = \frac{e^{z_i - z_{\max}}}{\sum_j e^{z_j - z_{\max}}}$$

This is mathematically equivalent (the $e^{-z_{\max}}$ cancels) but numerically stable since $z_i - z_{\max} \leq 0$.

**Temperature scaling** controls the sharpness of the distribution:

$$\text{softmax}(\mathbf{z}/\tau)_i = \frac{e^{z_i/\tau}}{\sum_j e^{z_j/\tau}}$$

- $\tau < 1$: sharper distribution (lower entropy, more deterministic)
- $\tau = 1$: standard softmax
- $\tau > 1$: softer distribution (higher entropy, more random)
- $\tau \to 0$: greedy/argmax selection
- $\tau \to \infty$: uniform distribution

Temperature is widely used in **sampling**, **knowledge distillation** (soft targets), and **calibration**.

In [ ]:
# Numerically stable softmax and temperature scaling

def softmax_naive(z):
    """Naive softmax — numerically unstable for large values."""
    exp_z = torch.exp(z)
    return exp_z / exp_z.sum()

def softmax_stable(z):
    """Numerically stable softmax using log-sum-exp trick."""
    z_shifted = z - z.max()  # subtract max for stability
    exp_z = torch.exp(z_shifted)
    return exp_z / exp_z.sum()

# Demo with normal logits
logits_normal = torch.tensor([2.0, 1.0, 0.5, -0.5, -1.0])
p_naive = softmax_naive(logits_normal)
p_stable = softmax_stable(logits_normal)
p_torch = F.softmax(logits_normal, dim=0)

print("Normal logits:", logits_normal.tolist())
print(f"Naive:   {p_naive.round(decimals=5).tolist()}")
print(f"Stable:  {p_stable.round(decimals=5).tolist()}")
print(f"PyTorch: {p_torch.round(decimals=5).tolist()}")
print(f"All match: {torch.allclose(p_stable, p_torch, atol=1e-6)}")

# Demo with LARGE logits — naive overflows
logits_large = torch.tensor([1000.0, 999.0, 998.0, 0.0, -100.0])
p_naive_large = softmax_naive(logits_large)
p_stable_large = softmax_stable(logits_large)
print(f"\nLarge logits: {logits_large.tolist()}")
print(f"Naive (overflows?): {p_naive_large.tolist()}")
print(f"Stable:             {p_stable_large.round(decimals=5).tolist()}")

# Temperature scaling
print("\n--- Temperature Scaling ---")
logits = torch.tensor([3.0, 1.5, 1.0, 0.5, 0.2, 0.1, 0.05, 0.01])
vocab = [f'tok{i}' for i in range(len(logits))]

print(f"{'tau':<8} {'max_prob':<12} {'entropy(bits)':<16} {'distribution'}")
print("-" * 70)
for tau in [0.1, 0.5, 1.0, 2.0]:
    p = F.softmax(logits / tau, dim=0)
    H = -(p * torch.log2(p.clamp(min=1e-10))).sum().item()
    print(f"{tau:<8.1f} {p.max().item():<12.4f} {H:<16.4f} {p.round(decimals=3).tolist()}")

## 3.8 Top-k & Top-p (Nucleus) Sampling

Pure temperature sampling can still assign probability to highly unlikely tokens. **Top-k** and **top-p** sampling truncate the distribution to only the most probable tokens before renormalizing.

**Top-k sampling:**
Keep only the $k$ tokens with highest probability, set all others to $-\infty$ (before softmax), then renormalize.

$$\tilde{z}_i = \begin{cases} z_i & \text{if } i \in \text{top-}k(z) \\ -\infty & \text{otherwise} \end{cases}$$

**Top-p (nucleus) sampling:**
Sort tokens by descending probability and keep the smallest set $S$ such that:

$$\sum_{i \in S} p_i \geq p$$

The nucleus adapts its size: when the model is confident (peaked distribution), fewer tokens are kept; when uncertain (flat distribution), more are included.

**Comparison:**
- Top-k is simple but uses a fixed number of tokens regardless of distribution shape
- Top-p is adaptive and generally produces better text quality
- Both can be combined: top-p applied after top-k

Typical values: $k \in \{40, 50\}$, $p \in \{0.9, 0.95\}$.

In [ ]:
# Top-k and Top-p (nucleus) sampling

torch.manual_seed(42)

def top_k_sample(logits, k, temperature=1.0):
    """
    Top-k sampling: keep k highest logits, mask the rest.
    Returns sampled token index.
    """
    # Apply temperature
    logits = logits / temperature
    # Find the k-th largest value
    topk_values, _ = torch.topk(logits, k)
    threshold = topk_values[-1]  # smallest value among top-k
    # Mask logits below threshold
    filtered_logits = logits.masked_fill(logits < threshold, float('-inf'))
    probs = F.softmax(filtered_logits, dim=-1)
    return torch.multinomial(probs, num_samples=1).item(), probs

def top_p_sample(logits, p, temperature=1.0):
    """
    Top-p (nucleus) sampling: keep smallest set of tokens with cumulative prob >= p.
    Returns sampled token index.
    """
    logits = logits / temperature
    probs = F.softmax(logits, dim=-1)
    # Sort by descending probability
    sorted_probs, sorted_indices = torch.sort(probs, descending=True)
    cumulative_probs = torch.cumsum(sorted_probs, dim=-1)
    # Remove tokens where cumulative prob exceeds p
    # Shift right so we include the token that crosses the threshold
    sorted_indices_to_remove = cumulative_probs - sorted_probs > p
    sorted_probs[sorted_indices_to_remove] = 0.0
    # Restore original order
    probs_filtered = torch.zeros_like(probs)
    probs_filtered.scatter_(0, sorted_indices, sorted_probs)
    # Renormalize
    probs_filtered = probs_filtered / probs_filtered.sum()
    return torch.multinomial(probs_filtered, num_samples=1).item(), probs_filtered

# Demo on random logits
vocab_size = 15
logits = torch.randn(vocab_size) * 2.0  # more spread out
probs_full = F.softmax(logits, dim=0)

print("Full distribution (sorted descending):")
sorted_p, sorted_idx = torch.sort(probs_full, descending=True)
for i, (idx, prob) in enumerate(zip(sorted_idx[:8].tolist(), sorted_p[:8].tolist())):
    cumsum = sorted_p[:i+1].sum().item()
    print(f"  rank {i+1}: token {idx:2d}  prob={prob:.4f}  cumsum={cumsum:.4f}")

# Top-k
k = 5
token_k, probs_k = top_k_sample(logits, k=k)
active_k = (probs_k > 0).sum().item()
print(f"\nTop-k (k={k}): sampled token={token_k}, active tokens={active_k}")
print(f"  Top-k probs: {probs_k[probs_k > 0].round(decimals=4).tolist()}")

# Top-p
p = 0.9
token_p, probs_p = top_p_sample(logits, p=p)
active_p = (probs_p > 0).sum().item()
print(f"\nTop-p (p={p}): sampled token={token_p}, active tokens={active_p}")
print(f"  Nucleus probs: {probs_p[probs_p > 0].round(decimals=4).tolist()}")

# Show how nucleus size adapts
print("\n--- Nucleus size for varying distribution peakedness ---")
for scale in [0.5, 1.0, 2.0, 4.0]:
    logits_scaled = torch.randn(vocab_size) * scale
    _, probs_filtered = top_p_sample(logits_scaled, p=0.9)
    active = (probs_filtered > 0).sum().item()
    print(f"  logit_scale={scale:.1f}: nucleus size = {active} tokens")

## 3.9 Perplexity Evaluation

**Perplexity** is the standard intrinsic metric for language model quality:

$$\text{PPL}(\mathcal{X}) = \exp\!\left(-\frac{1}{T}\sum_{t=1}^{T} \log P(x_t \mid x_{<t})\right)$$

It can be interpreted as the **effective branching factor** — if PPL = $b$, the model is as uncertain at each step as a uniform distribution over $b$ equally likely tokens.

**Lower perplexity = better model** (more confident about correct predictions).

**Practical considerations:**
- PPL is sensitive to tokenization: character-level vs BPE models have different baselines
- Typically evaluated on a held-out test set
- GPT-2 small: ~35 PPL on WikiText-103; GPT-4 class models: <10 PPL on standard benchmarks
- **Stride evaluation:** for long documents, use a sliding window to avoid treating padding as context

**Bits per character (BPC)** is an equivalent metric in natural units: $\text{BPC} = \text{PPL}^{1/L}$ where $L$ is avg characters per token.

In [ ]:
# Perplexity evaluation on a toy language model

torch.manual_seed(42)

class ToyLM(nn.Module):
    """Minimal language model: embedding -> linear -> logits."""
    def __init__(self, vocab_size, embed_dim, hidden_dim):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim)
        self.fc = nn.Linear(embed_dim, hidden_dim)
        self.out = nn.Linear(hidden_dim, vocab_size)
    
    def forward(self, x):
        # x: (B, T) token ids
        emb = self.embedding(x)  # (B, T, E)
        h = torch.tanh(self.fc(emb))  # (B, T, H)
        logits = self.out(h)  # (B, T, V)
        return logits

vocab_size = 50
embed_dim = 16
hidden_dim = 32
seq_len = 20

model = ToyLM(vocab_size, embed_dim, hidden_dim)

# Generate a toy token sequence
token_sequence = torch.randint(0, vocab_size, (1, seq_len + 1))  # (1, T+1)
inputs = token_sequence[:, :-1]   # (1, T) — context
targets = token_sequence[:, 1:]   # (1, T) — next tokens to predict

# Forward pass
with torch.no_grad():
    logits = model(inputs)  # (1, T, V)

# Compute per-token log-probabilities
log_probs = F.log_softmax(logits, dim=-1)  # (1, T, V)
target_log_probs = log_probs.gather(
    dim=-1, index=targets.unsqueeze(-1)
).squeeze(-1)  # (1, T)

# Average negative log-likelihood
avg_nll = -target_log_probs.mean()
perplexity = torch.exp(avg_nll)

print(f"Sequence length:         {seq_len}")
print(f"Vocabulary size:         {vocab_size}")
print(f"Random baseline PPL:     {vocab_size:.2f}")
print(f"Average NLL:             {avg_nll.item():.4f}")
print(f"Perplexity:              {perplexity.item():.4f}")

# Per-position perplexity
per_pos_ppl = torch.exp(-target_log_probs.squeeze(0))
print(f"\nPer-position perplexity (first 10 tokens):")
print(f"  {per_pos_ppl[:10].round(decimals=2).tolist()}")
print(f"  Mean: {per_pos_ppl.mean().item():.2f}")

# Cross-entropy loss matches
ce_loss = nn.CrossEntropyLoss()(logits.view(-1, vocab_size), targets.view(-1))
ppl_from_ce = torch.exp(ce_loss)
print(f"\nPPL from nn.CrossEntropyLoss: {ppl_from_ce.item():.4f}")
print(f"PPL from manual NLL:          {perplexity.item():.4f}")
print(f"Match: {abs(ppl_from_ce.item() - perplexity.item()) < 0.01}")

# Show PPL for different 'quality' models (simulated)
print("\n--- PPL interpretation ---")
for nll in [0.5, 1.0, 2.0, 3.0, 3.91, 5.0]:
    ppl = math.exp(nll)
    print(f"  NLL={nll:.2f} -> PPL={ppl:7.2f}  ~ uniform over {ppl:.0f} tokens")

## Chapter 3 Summary

| Concept | Formula | LLM Role |
|---------|---------|----------|
| Chain rule | $P(x_{1:T}) = \prod_t P(x_t \mid x_{<t})$ | Autoregressive generation |
| MLE | $\arg\max \sum \log p(x_i; \theta)$ | Pretraining objective |
| Entropy | $H = -\sum p \log p$ | Measures uncertainty |
| Cross-entropy | $\mathcal{L} = -\sum y_i \log \hat{p}_i$ | Training loss |
| Perplexity | $\exp(\mathcal{L})$ | Evaluation metric |
| KL divergence | $\sum P \log(P/Q)$ | RLHF penalty |
| Temperature | $\text{softmax}(z/\tau)$ | Sampling sharpness |
| Top-k | Keep $k$ highest logits | Decoding strategy |
| Top-p | Smallest set with cumsum $\geq p$ | Adaptive decoding |

**Key takeaways:**
- Every LLM training step minimizes cross-entropy = negative log-likelihood
- Perplexity is the exponential of cross-entropy — lower is better
- KL divergence appears in RLHF to prevent reward hacking
- Temperature, top-k, and top-p control the quality/diversity tradeoff in generation

**Next:** Chapter 4 covers the transformer architecture — attention, positional encoding, and the full forward pass.